# End-to-end serving benchmark: SparseWalker vs SASRec

Measures one new-event request on A100, batch=1.

**SASRec:** exact 2-layer incremental KV-cache update -> dense catalog scoring/top-10.

**SparseWalker:** Triton local Walker update -> 2 real Triton temporal SWG reads -> temporal projection/FFNs -> Triton sparse terminal retrieval/top-10.

Historical KV caches, temporal graph adjacency/K/V, concept graph, and terminal support are persistent serving state and are not rebuilt inside the timed request. Temporal graph insertion/index maintenance is also excluded from the query critical path. This is a GPU query-path microbenchmark, not a production p99 claim.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, torch
REPO='/content/Sparsewalker'
BRANCH='agent/serving-speed-benchmark'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
sys.path.insert(0,f'{REPO}/src')
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU',torch.cuda.get_device_name(0),'bf16',torch.cuda.is_bf16_supported())
print('BRANCH',BRANCH)


## Quick benchmark first

Runs history lengths 200 / 1k / 10k and catalogs 3.7k / 100k / 1M. The first combination includes compilation warmup, so only the printed `E2E` timings are the timed steady-state values.

In [ ]:
import runpy, sys
SCRIPT=f'{REPO}/benchmarks/run_e2e_serving_speed.py'
sys.argv=[SCRIPT,'--lengths','200','1000','10000','--catalogs','3706','100000','1000000','--beam','16','--hops','4']
print('E2E SERVING BENCH START',flush=True)
runpy.run_path(SCRIPT,run_name='__main__')
print('E2E SERVING BENCH END',flush=True)


## Compact result

Paste the `E2E` lines back into ChatGPT.

In [ ]:
import json
from pathlib import Path
p=Path('/content/drive/MyDrive/sparsewalker_speed/e2e_serving_speed.json')
if p.exists():
    r=json.loads(p.read_text())
    for row in r['rows']:
        print(row)
